# NGS PDF 解析測試
測試 `L9534589_K52600039.pdf` 透過現有 API / Service 的解析結果

In [ ]:
import json, re, sys, pprint
import requests
import pdfplumber

sys.path.insert(0, ".")

PDF_PATH = "L9534589_K52600039.pdf"
BASE_URL = "http://localhost:8591"

## 1. PDF 原始文字（pdfplumber 能抽出什麼）

In [ ]:
with pdfplumber.open(PDF_PATH) as pdf:
    pages_text = [p.extract_text() or "" for p in pdf.pages]
    total_pages = len(pdf.pages)

print(f"總頁數: {total_pages}，總字數: {sum(len(t) for t in pages_text)}")
for i, t in enumerate(pages_text):
    print(f"\n=== Page {i+1} ({len(t)} chars) ===")
    print(t[:600] if t else "(空)")

## 2. Rule-based parser（POST /convert/pdf）

In [ ]:
with open(PDF_PATH, "rb") as f:
    resp = requests.post(
        f"{BASE_URL}/convert/pdf",
        files={"file": (PDF_PATH, f, "application/pdf")},
    )

print(f"Status: {resp.status_code}")
result_rule = resp.json()

print(f"  patient_name : {result_rule.get('patient_name')}")
print(f"  patient_id   : {result_rule.get('patient_id')}")
print(f"  report_date  : {result_rule.get('report_date')}")
print(f"  panel_name   : {result_rule.get('panel_name')}")
print(f"  specimen_type: {result_rule.get('specimen_type')}")
print(f"  conclusion   : {result_rule.get('conclusion')}")
print(f"  variants 數量: {len(result_rule.get('variants', []))}")
for v in result_rule.get("variants", []):
    pprint.pprint(v)

## 3. LLM Service（llm_service.parse_ngs_report）
走逐頁解析＋快取，結果會存在 `data/cache/pdf_pages/`

In [ ]:
from app.llm.service import llm_service

result_llm = llm_service.parse_ngs_report(PDF_PATH)

print(f"  patient_name : {result_llm.get('patient_name')}")
print(f"  patient_id   : {result_llm.get('patient_id')}")
print(f"  report_date  : {result_llm.get('report_date')}")
print(f"  panel_name   : {result_llm.get('panel_name')}")
print(f"  specimen_type: {result_llm.get('specimen_type')}")
print(f"  ref_genome   : {result_llm.get('ref_genome')}")
print(f"  conclusion   : {result_llm.get('conclusion')}")
print(f"  variants 數量: {len(result_llm.get('variants', []))}")
for v in result_llm.get("variants", []):
    pprint.pprint(v)

## 4. Debug：逐頁看 LLM 原始回應

In [ ]:
from langchain_community.llms import Ollama
from app.llm.service import _PROMPT_FIRST_PAGE, _PROMPT_OTHER_PAGE, _parse_json
from app.config import get_settings

settings = get_settings()
llm = Ollama(model=settings.llm_model, base_url=settings.llm_base_url, temperature=0.1)

raw_responses = []   # 存每頁原始回應
parsed_pages  = []   # 存每頁解析結果

for i, page_text in enumerate(pages_text):
    template = _PROMPT_FIRST_PAGE if i == 0 else _PROMPT_OTHER_PAGE
    prompt = template.format(page_text=page_text)

    print(f"\n{'='*60}")
    print(f"Page {i+1}/{total_pages} — 呼叫 LLM...")
    raw = llm.invoke(prompt)
    raw_responses.append(raw)

    parsed = _parse_json(raw)
    parsed_pages.append(parsed)

    print(f"--- 原始回應 ---\n{raw[:800]}")
    print(f"--- 解析結果 ---")
    pprint.pprint(parsed)

## 5. 合併所有頁面結果

In [ ]:
from app.llm.service import _merge_pages, _empty_result

valid_pages = [p for p in parsed_pages if p]
merged = _merge_pages(valid_pages) if valid_pages else _empty_result()

print("合併結果：")
pprint.pprint({k: v for k, v in merged.items() if k != "variants"})
print(f"\nvariants 共 {len(merged['variants'])} 筆：")
for v in merged["variants"]:
    pprint.pprint(v)